In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score

# Data and features
train=pd.read_csv("MLProject2026/data/case1/train1.csv")
test=pd.read_csv("MLProject2026/data/case1/test1.csv")
for df in (train, test):
    df["n"] = df["speed_rpm"] / 1000
    df["Q"] = df["flow_m3_h"]
    df["P"] = df["power_w"]
    df["deltaP"]=df["outlet_pressure_bar"]-df["inlet_pressure_bar"]

X_train, y_train= train[["n","P","deltaP"]], train["Q"]
X_test, y_test= test[["n","P","deltaP"]], test["Q"]

In [2]:
all_cols = list(X_train.columns)
n_cols = len(all_cols)

all_subsets = [[]]
for col in all_cols:
    new_subsets = []
    for subset in all_subsets:
        new_subsets.append(subset + [col])
    all_subsets = all_subsets + new_subsets
all_subsets = all_subsets[1:]

In [3]:
cv = KFold(n_splits=5, shuffle=True, random_state=30)

results = []
for subset in all_subsets:
    scores = cross_val_score(LinearRegression(), X_train[subset], y_train,
                             cv=cv, scoring="neg_mean_squared_error")
    e = -scores
    results.append({"n_terms": len(subset), "features": tuple(subset),
                    "mse": e.mean(), "se": e.std(ddof=1) / np.sqrt(len(e))})
results = pd.DataFrame(results)
results.sort_values("mse")

,n_terms,features,mse,se
6,3,"(n, P, deltaP)",12.192435,0.797782
5,2,"(P, deltaP)",20.130007,0.689996
4,2,"(n, deltaP)",34.991711,1.238502
2,2,"(n, P)",88.123316,1.974420
1,1,"(P,)",100.152153,2.288870
0,1,"(n,)",132.518247,4.362363
3,1,"(deltaP,)",171.359433,7.318371


In [4]:
model=LinearRegression().fit(X_train,y_train)
y_pred=model.predict(X_test)

r2_score(y_test, y_pred)


0.9141984630856215

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score

# Data and features
train=pd.read_csv("MLProject2026/data/case1/train1.csv")
test=pd.read_csv("MLProject2026/data/case1/test1.csv")
for df in (train, test):
    df["n"] = df["speed_rpm"] / 1000
    df["Q/n"] = df["flow_m3_h"]/df["n"]
    df["P/n^3"] = df["power_w"]/(df["n"]**3)
    df["deltaP/n^2"] = (df["outlet_pressure_bar"] - df["inlet_pressure_bar"]) / (df["n"]**2)

X_train, y_train= train[["n","P/n^3","deltaP/n^2"]], train["Q/n"]
X_test, y_test= test[["n","P/n^3","deltaP/n^2"]], test["Q/n"]

In [6]:
all_cols = list(X_train.columns)
n_cols = len(all_cols)

all_subsets = [[]]
for col in all_cols:
    new_subsets = []
    for subset in all_subsets:
        new_subsets.append(subset + [col])
    all_subsets = all_subsets + new_subsets
all_subsets = all_subsets[1:]

In [7]:
cv = KFold(n_splits=5, shuffle=True, random_state=30)

results = []
for subset in all_subsets:
    scores = cross_val_score(LinearRegression(), X_train[subset], y_train,
                             cv=cv, scoring="neg_mean_squared_error")
    e = -scores
    results.append({"n_terms": len(subset), "features": tuple(subset),
                    "mse": e.mean(), "se": e.std(ddof=1) / np.sqrt(len(e))})
results = pd.DataFrame(results)
results.sort_values("mse")

,n_terms,features,mse,se
6,3,"(n, P/n^3, deltaP/n^2)",0.543541,0.066914
5,2,"(P/n^3, deltaP/n^2)",1.160866,0.084072
3,1,"(deltaP/n^2,)",2.313254,0.164355
4,2,"(n, deltaP/n^2)",2.315409,0.163350
2,2,"(n, P/n^3)",5.877787,0.275991
1,1,"(P/n^3,)",7.877513,0.527802
0,1,"(n,)",12.012569,0.404248


In [8]:
model=LinearRegression().fit(X_train,y_train)
y_pred=model.predict(X_test)
Q_pred = y_pred * test["n"]
r2_score(test["flow_m3_h"], Q_pred)

0.9708108527492016